# 02 — Content Intelligence SQL

**Engine:** DuckDB (in-memory)  
**Scope:** 10 business SQL queries on live TMDB-schema data


In [ ]:
import pandas as pd
import duckdb
import matplotlib.pyplot as plt
import json

# Load data
trending = pd.read_csv("../data/trending_movies_latest.csv")
tv = pd.read_csv("../data/popular_tv_latest.csv")
genres = pd.read_csv("../data/movie_genres_latest.csv")
genre_pop = pd.read_csv("../data/genre_popularity_latest.csv")
upcoming = pd.read_csv("../data/upcoming_movies_latest.csv")
top_rated = pd.read_csv("../data/top_rated_movies_latest.csv")

def parse_genres(s):
    try: return json.loads(s) if isinstance(s, str) else []
    except: return []

trending["genre_ids_list"] = trending["genre_ids"].apply(parse_genres)
tv["genre_ids_list"] = tv["genre_ids"].apply(parse_genres)

con = duckdb.connect(":memory:")
con.register("trending_movies", trending)
con.register("popular_tv", tv)
con.register("movie_genres", genres)
con.register("genre_popularity", genre_pop)
con.register("upcoming_movies", upcoming)
con.register("top_rated_movies", top_rated)

con.execute("CREATE TABLE genre_lookup AS SELECT genre_id, genre_name FROM movie_genres")
print("Tables registered in DuckDB")


## Q1: Content Mix — Movies vs TV Shows

In [ ]:
SELECT 'Movies' AS content_type, COUNT(*) AS total FROM trending_movies
UNION ALL
SELECT 'TV Shows', COUNT(*) FROM popular_tv;


## Q2: Top 10 Genres by Title Count

In [ ]:
SELECT genre_name, COUNT(*) AS title_count
FROM (
    SELECT UNNEST(genre_ids_list) AS gid FROM trending_movies
) t
JOIN genre_lookup gl ON t.gid = gl.genre_id
GROUP BY genre_name
ORDER BY title_count DESC
LIMIT 10;


![Q2: Top 10 Genres by Title Count](../figures/02_q2_top_genres.png)

## Q3: Average Rating by Genre (min 50 titles)

In [ ]:
SELECT genre_name, ROUND(AVG(vote_average), 2) AS avg_rating, COUNT(*) AS title_count
FROM (
    SELECT vote_average, UNNEST(genre_ids_list) AS gid FROM trending_movies
) t
JOIN genre_lookup gl ON t.gid = gl.genre_id
GROUP BY genre_name
HAVING COUNT(*) > 50
ORDER BY avg_rating DESC;


![Q3: Average Rating by Genre (min 50 titles)](../figures/02_q3_avg_rating_by_genre.png)

## Q4: Hidden Gems — High Rating, Low Popularity

In [ ]:
SELECT title, vote_average, popularity, release_date
FROM trending_movies
WHERE vote_average >= 8.0
  AND popularity < (SELECT quantile_cont(popularity, 0.5) FROM trending_movies)
ORDER BY vote_average DESC, popularity ASC
LIMIT 15;


## Q5: Release Year Trend

In [ ]:
SELECT CAST(SUBSTR(release_date, 1, 4) AS INTEGER) AS release_year,
       COUNT(*) AS movie_count,
       ROUND(AVG(vote_average), 2) AS avg_rating
FROM trending_movies
WHERE release_date IS NOT NULL AND release_date != ''
GROUP BY release_year
HAVING release_year BETWEEN 2000 AND 2021
ORDER BY release_year;


![Q5: Release Year Trend](../figures/02_q5_release_trend.png)

## Q6: Rating by Vote Count Bucket

In [ ]:
SELECT CASE
    WHEN vote_count < 100 THEN '0-99'
    WHEN vote_count < 1000 THEN '100-999'
    WHEN vote_count < 5000 THEN '1K-4.9K'
    ELSE '5K+'
END AS vote_bucket,
COUNT(*) AS titles,
ROUND(AVG(vote_average), 2) AS avg_rating,
ROUND(AVG(popularity), 2) AS avg_popularity
FROM trending_movies
GROUP BY vote_bucket
ORDER BY avg_rating DESC;


![Q6: Rating by Vote Count Bucket](../figures/02_q6_vote_bucket_rating.png)

## Q7: TV Show Decade Distribution

In [ ]:
SELECT CASE
    WHEN CAST(SUBSTR(release_date, 1, 4) AS INTEGER) < 1990 THEN '< 1990s'
    WHEN CAST(SUBSTR(release_date, 1, 4) AS INTEGER) < 2000 THEN '1990s'
    WHEN CAST(SUBSTR(release_date, 1, 4) AS INTEGER) < 2010 THEN '2000s'
    WHEN CAST(SUBSTR(release_date, 1, 4) AS INTEGER) < 2020 THEN '2010s'
    ELSE '2020s'
END AS decade,
COUNT(*) AS tv_count,
ROUND(AVG(vote_average), 2) AS avg_rating
FROM popular_tv
WHERE release_date IS NOT NULL AND release_date != ''
GROUP BY decade
ORDER BY decade;


## Q8: Upcoming Releases by Month

In [ ]:
SELECT SUBSTR(release_date, 6, 2) AS release_month,
       COUNT(*) AS upcoming_count,
       ROUND(AVG(vote_average), 2) AS avg_rating
FROM upcoming_movies
WHERE release_date IS NOT NULL AND release_date != ''
GROUP BY release_month
ORDER BY release_month;


## Q9: Genre Popularity Ranking

In [ ]:
SELECT genre_name, top_movie_popularity, total_movies_in_genre,
       ROUND(top_movie_popularity / NULLIF(total_movies_in_genre, 0), 2) AS pop_per_title
FROM genre_popularity
ORDER BY top_movie_popularity DESC
LIMIT 15;


## Q10: Content Quality Tiers

In [ ]:
SELECT CASE
    WHEN vote_average >= 8.0 THEN 'Excellent (8.0+)'
    WHEN vote_average >= 7.0 THEN 'Good (7.0–7.9)'
    WHEN vote_average >= 6.0 THEN 'Average (6.0–6.9)'
    ELSE 'Below Average (< 6.0)'
END AS quality_tier,
COUNT(*) AS title_count,
ROUND(AVG(popularity), 2) AS avg_popularity,
ROUND(AVG(vote_count), 0) AS avg_votes
FROM trending_movies
GROUP BY quality_tier
ORDER BY AVG(vote_average) DESC;


![Q10: Content Quality Tiers](../figures/02_q10_quality_tiers.png)

---
**Summary of Key Findings**
- Movies outnumber TV shows 6,131 to 2,676.
- Top genres: Drama, Comedy.
- Highest-rated genre: Drama (6.51 avg).
- 37.6% of movies are rated Good or Excellent.
